# ST7 Project 2026

## Algorithm
1. **[LONG]** Forward solve: m_n → SEM3D → u_sim(x_ric, t) and u(x,t)
2. Misfit: r(t) = u_sim(t) - d_obs(t)
3. **[LONG]** Adjoint solve: r(T-t) → SEM3D backward → Λ(x,t)
4. Gradient: g_λ, g_μ from cross-correlation of ε[u] and ε[Λ]
5. CG direction (Fletcher-Reeves): p_n = -g_n + β_n · p_{n-1}
6. Line search (backtracking Armijo): find α_n
7. Update: m_{n+1} = m_n + α_n · p_n

## 0. Imports

In [ ]:
import numpy as np
import h5py
import os
import subprocess
import sys
import importlib
from collections import deque
from pysem import parse_sem3d_traces

# pysem toolkit
from pysem.parse_sem3d_traces import ParseSEM3DH5Traces
from pysem.generate_h5_materials import write_h5
from util_func.sbatch_and_wait import sbatch_and_wait
from util_func.compute_misfit import compute_misfit
from util_func.read_stations_pos import read_stations_pos
from util_func.write_backward_spec import write_backward_spec_from_template
from util_func.write_misfit_files import write_time_reversed_residual_files
from util_func.dir_research_idea import compute_dir
from util_func.modify_h5_materials_greg import n_elems, grid_to_vec, vec_to_grid, modify_h5

## 1. Paths and parameters

In [ ]:
os.mkdir("")

In [ ]:
# --- Paths ---
# WKDIR: working directory where SEM3D is launched
# must contain: input.spec, material.spec, stations.txt, gaussian_stf.txt, sem/mesh4spec.*.h5

TRACES_SIMULATED_FOLDER_PATH = ""
TRACES_OSSERVATED_PATH = "/usr/users/cea_seism/benede_gio/CEISM-Project/Uobs"

ADJOINT_DIR = "/path/to/adjoint_case"
MISFIT_DIR = os.path.join(ADJOINT_DIR, "adjoint_sources")
STATIONS_FILE_PATH = "/usr/users/cea_seism/tran_ngo/tutorials_SEM3D_DCE/tutorial2/stations.txt"
BACKWARD_INPUT_SPEC_TEMPLATE_PATH = "/path/to/input.spec"
BACKWARD_INPUT_SPEC_OUTPUT_PATH = ""


# D_OBS_DIR: directory containing observed data (tutorial2/prot/Protection_.../Capteurs/)

# --- Initial material parameters m_0 ---
# domain limits, discretization, initial gradient

# --- Optimization parameters ---
# N_ITER: maximum number of RTM iterations
# TOL: gradient convergence tolerance
# ALPHA_0: initial step size for line search
# ARMIJO_C: backtracking reduction constant
# ARMIJO_TAU: sufficient decrease factor

## 2. Load observed data d_obs

In [ ]:
obs_stream = ParseSEM3DH5Traces(
    wkdir=TRACES_OSSERVATED_PATH,
    format='h5',
    names=['Uobs'],
    variables=['Displ'],
    components=['x', 'y', 'z']
)

obs_monitor = obs_stream['Uobs']
obs_u = obs_monitor.data['Displ']

## 3. Initial material m_0

In [ ]:
# Generate initial material HDF5 files (e.g. homogeneous model)
# output: example_la.h5, example_mu.h5, example_ds.h5 in WKDIR

# m_la: 3D array of Lambda
# m_mu: 3D array of Mu
# m_ds: 3D array of Rho (fixed, not inverted)

## 4. Algorithm

In [ ]:
# CG variable initialization
# p_prev = 0  (previous direction)
# g_prev = 0  (previous gradient)
# J_prev = inf
N_ITER = 10 #to be modified

# /============ Backtracking line search initialization before the loops ============\ #
M = 5    # parameter of L-BFGS
Y_LA, S_LA, Y_MU, S_MU = deque([None for _ in range(M)]), deque([None for _ in range(M)]), deque([None for _ in range(M)]), deque([None for _ in range(M)])
GRAD_LA , GRAD_MU , LA , MU = deque([None, None]), deque([None, None]), deque([None, None]), deque([None, None])
materials_paths = {'Mu': ... ### [ENTER PATH where mu.h5 is stored (e.g. : "/usr/users/cea_seism/benede_gio/CEISM-Project/materials/example_la.h5")]
                 ,'La': ... ### [ENTER PATH where la.h5 is stored]
}
Nx,Ny,Nz = n_elems(file_paths = materials_paths)
# \                                                                                  / #         

for n in range(N_ITER):

    # ── STEP 1 ────────────────────────────────────────
    sbatch_and_wait("MESHER.sbatch")
    sbatch_and_wait("SOLVER.sbatch")

    # ── STEP 2: MISFIT ────────────────────────────────────────────────────

    J, residual, t_sim, dt_sim = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, obs_u)
    
    # ── STEP 3 ────────────────────────────────────────

    time_reversed_residual = residual[::-1, :, :]

    stations = read_stations_pos(STATIONS_FILE_PATH)

    file_names = write_time_reversed_residual_files(time_reversed_residual, 
                                                    t_sim, 
                                                    OUTPUT_DIR=MISFIT_DIR)

    write_backward_spec_from_template(template_backward_spec_path=BACKWARD_INPUT_SPEC_TEMPLATE_PATH, 
                                      output_backward_spec_path = BACKWARD_INPUT_SPEC_OUTPUT_PATH, 
                                      stations = stations, 
                                      file_names = file_names, 
                                      misfit_rel_dir="adjoint_sources")

    sbatch_and_wait("SOLVER.sbatch")

    #input.spec
    #material

    # ── STEP 4: GRADIENT ─────────────────────────────────────────────────
    # Read edev and evol of u(x,t) from forward snapshots with ParseSEM3DSnapshots
    # Read edev and evol of Λ(x,t) from adjoint snapshots with ParseSEM3DSnapshots
    # Integrate in time (sum over all timesteps):
    #   g_la(x) = sum_t  evol[u](x,t) * evol[Λ](x,t)   (gradient w.r.t. Lambda)
    #   g_mu(x) = sum_t  edev[u](x,t) : edev[Λ](x,t)   (gradient w.r.t. Mu)
    # Add regularization term if present

    # Convergence check
    # if ||g|| < TOL: break

    ### ======================================== STEP 9 ======================================== ###
    la, mu = ... , ...  ### Retrieve from STEP 5 (of the slides) and CONVERT INTO a 1D NP.ARRAY
    grad_la, grad_mu = ... , ... # Retrieve from previous steps (Long) and CONVERT INTO a 1D NP.ARRAY
    
    LA.pop(), MU.pop()
    LA.appendleft(la), MU.appendleft(mu)
    GRAD_LA.pop(), GRAD_MU.pop()
    GRAD_LA.appendleft(grad_la), GRAD_MU.appendleft(grad_mu)

    y_la, y_mu , s_la, s_mu = GRAD_LA[0] - GRAD_LA[1] , GRAD_MU[0] - GRAD_MU[1] , LA[0] - LA[1] , MU[0] - MU[1]
    Y_LA.pop(), Y_MU.pop()
    Y_LA.appendleft(y_la), Y_MU.appendleft(y_mu)
    S_LA.pop(), S_MU.pop()
    S_LA.appendleft(s_la), S_MU.appendleft(s_mu)

    dir_la , dir_mu = compute_dir(new_y = grad_la, y_queue = Y_LA, s_queue = S_LA, M = M) , compute_dir(new_y = grad_mu, y_queue = Y_MU, s_queue = S_MU, M = M)

    ### ======================================== STEP 10 ======================================= ###
    alpha_la=1, alpha_mu=1 , c1=1e-4 , xi=0.5
    J_thresh = J + c1(alpha_la*np.dot(dir_la,grad_la) + alpha_mu*np.dot(dir_mu,grad_mu))
    J_learn = np.inf

    vec_to_add_la, vec_to_add_mu = 2*alpha_la*dir_la , 2*alpha_mu*dir_mu
    modify_h5(materials_paths['La'],vec_to_add_la,Nx,Ny,Nz)
    modify_h5(materials_paths['Mu'],vec_to_add_mu,Nx,Ny,Nz)

    ## Backtracking loop ##
    while J_learn >= J_thresh:
        vec_to_add_la -= alpha_la*dir_la
        vec_to_add_mu -= alpha_mu*dir_mu
        modify_h5(materials_paths['La'],vec_to_add_la,Nx,Ny,Nz)
        print("la.h5 modified",flush=True)
        modify_h5(materials_paths['Mu'],vec_to_add_mu,Nx,Ny,Nz)
        print("mu.h5 modified",flush=True)
        # Launch SEM3D ('/workdir/match/la.h5','/workdir/match/mu.h5' have just been updated
        # Call sbatch SOLVER, and then get the traces and Uobs
        sbatch_and_wait("SOLVER.sbatch")
        J_learn = compute_misfit(TRACES_SIMULATED_FOLDER_PATH,obs_u)
        alpha_la *= xi
        alpha_mu *= xi